In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

class DebuggableMultiHeadAttention(nn.Module):
    def __init__(self, embed_dim, num_heads):
        super().__init__()
        assert embed_dim % num_heads == 0, "Embedding dimension must be divisible by num_heads"
        
        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads
        
        # Linear layers for projecting input embeddings
        self.q_proj = nn.Linear(embed_dim, embed_dim)
        self.k_proj = nn.Linear(embed_dim, embed_dim)
        self.v_proj = nn.Linear(embed_dim, embed_dim)
        
        # Final output projection
        self.out_proj = nn.Linear(embed_dim, embed_dim)
        
    def forward(self, query, key, value, mask=None, debug=False):
        batch_size, seq_len, _ = query.shape
        
        # 1. Linear projections
        # Shape: [B, N, C]
        q = self.q_proj(query)
        k = self.k_proj(key)
        v = self.v_proj(value)
        
        # 2. Reshape and transpose for multi-head processing
        # Target Shape: [B, num_heads, N, head_dim]
        q = q.view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        k = k.view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        v = v.view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        
        # 3. Calculate Pre-Softmax raw energy scores
        # Formula: Q * K^T / sqrt(d_k)
        # Shape: [B, num_heads, N, N]
        scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(self.head_dim)
        
        # Apply attention mask if provided (e.g., causal mask or padding mask)
        if mask is not None:
            scores = scores.masked_fill(mask == 0, float('-inf'))
            
        # 4. Calculate Attention Weights (probabilities)
        # Shape: [B, num_heads, N, N]
        attn_weights = F.softmax(scores, dim=-1)
        
        # 5. Optional Debug Hook: Print or log internal values
        if debug:
            print("\n" + "="*50)
            print("--- INTERNALS FOR LEARNING & OBSERVATION ---")
            print(f"Q Projected Stats: Mean={q.mean().item():.4f}, Std={q.std().item():.4f}")
            print(f"K Projected Stats: Mean={k.mean().item():.4f}, Std={k.std().item():.4f}")
            print(f"Raw Pre-Softmax Scores (Max/Min): {scores.max().item():.4f} / {scores.min().item():.4f}")
            print(f"Attention Weights Range: {attn_weights.max().item():.4f} to {attn_weights.min().item():.4f}")
            print("\nSample Attention Matrix for Head 0, Batch 0:\n", attn_weights[0, 0].detach().cpu().numpy().round(3))
            print("="*50 + "\n")
            
        # 6. Apply weights to values
        # Shape: [B, num_heads, N, head_dim]
        context = torch.matmul(attn_weights, v)
        
        # 7. Concatenate heads back into original channel dimension
        # Shape: [B, N, C]
        context = context.transpose(1, 2).contiguous().view(batch_size, seq_len, self.embed_dim)
        
        # 8. Output projection
        output = self.out_proj(context)
        
        # Return output along with internal values for analysis/plotting
        return output, attn_weights, scores
